# 03 - Real Data: DermaMNIST
Goal: load real dermatoscope images, understand the samples and labels, see the class imbalance,
and build a Dataset + DataLoader that feeds the model in batches.

In [1]:
from medmnist import DermaMNIST

# download=True fetches it once; 28x28 images to start (CPU-friendly)
train_data = DermaMNIST(split="train", download=True, size=28)

print(train_data)

100%|██████████| 19.7M/19.7M [00:09<00:00, 2.16MB/s]


Dataset DermaMNIST of size 28 (dermamnist)
    Number of datapoints: 7007
    Root location: C:\Users\study\.medmnist
    Split: train
    Task: multi-class
    Number of channels: 3
    Meaning of labels: {'0': 'actinic keratoses and intraepithelial carcinoma', '1': 'basal cell carcinoma', '2': 'benign keratosis-like lesions', '3': 'dermatofibroma', '4': 'melanoma', '5': 'melanocytic nevi', '6': 'vascular lesions'}
    Number of samples: {'train': 7007, 'val': 1003, 'test': 2005}
    Description: The DermaMNIST is based on the HAM10000, a large collection of multi-source dermatoscopic images of common pigmented skin lesions. The dataset consists of 10,015 dermatoscopic images categorized as 7 different diseases, formulized as a multi-class classification task. We split the images into training, validation and test set with a ratio of 7:1:2. The source images of 3×600×450 are resized into 3×28×28.
    License: CC BY-NC 4.0


## What DermaMNIST is
- 7007 train / 1003 val / 2005 test images, split 7:1:2 (val = held-out "real exam")
- RGB images, shape (3, 28, 28), so 3 x 28 x 28 = 2352 input values (not 784 - TinyNet needs updating)
- 7 classes = skin lesion diagnoses. Class 5 (nevi) common, class 4 (melanoma) rare -> class imbalance
- Source: HAM10000 dermatoscopy dataset

In [2]:
import numpy as np

labels = train_data.labels.flatten()   # the label for every training image
classes, counts = np.unique(labels, return_counts=True)

for c, n in zip(classes, counts):
    pct = 100 * n / len(labels)
    print(f"class {c}: {n:5d}  ({pct:4.1f}%)")

class 0:   228  ( 3.3%)
class 1:   359  ( 5.1%)
class 2:   769  (11.0%)
class 3:    80  ( 1.1%)
class 4:   779  (11.1%)
class 5:  4693  (67.0%)
class 6:    99  ( 1.4%)


## Class imbalance (the core challenge)
Counts: class 5 = 67%, class 3 = 1.1%. A 59x gap between most and least common.
- A model that ALWAYS predicts class 5 gets 67% accuracy while learning nothing -> accuracy is misleading here.
- The rare classes include class 4 (melanoma), the most dangerous, so ignoring rare classes is exactly the wrong failure.
- Plan: judge by per-class recall / F1 (not accuracy), and fight imbalance with weighted loss, then measure what helps.

## Dataset and DataLoader
- Dataset = fetches ONE sample (image + label) by index. DermaMNIST already is one.
- DataLoader = wraps a Dataset and yields BATCHES, shuffled each epoch. The training loop iterates over it.
- We feed data in small batches (e.g. 64), not all 7007 at once: less memory, better training.
- A transform converts each PIL image to a tensor as it is loaded (the model needs tensors, not PIL images).

In [1]:
import torch
from torchvision import transforms
from torch.utils.data import DataLoader
from medmnist import DermaMNIST

transform = transforms.Compose([
    transforms.ToTensor(),                       # PIL image -> tensor, scales pixels 0-255 into 0.0-1.0
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),   # shift pixels to roughly -1..1
])

train_data = DermaMNIST(split="train", download=True, size=28, transform=transform)
val_data   = DermaMNIST(split="val",   download=True, size=28, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=64, shuffle=False)

print("train batches:", len(train_loader), " val batches:", len(val_loader))

train batches: 110  val batches: 16


## Batch count
7007 / 64 = 109.5 -> 110 batches (last batch has only 63 images, not 64).
This is why we never hardcode batch size: `x.size(0)` handles the ragged last batch.

In [2]:
images, labels = next(iter(train_loader))   # grab the first batch

print("images shape:", images.shape)
print("labels shape:", labels.shape)
print("labels dtype:", labels.dtype)
print("pixel min/max:", images.min().item(), images.max().item())

images shape: torch.Size([64, 3, 28, 28])
labels shape: torch.Size([64, 1])
labels dtype: torch.int64
pixel min/max: -1.0 1.0


## One batch, inspected
- images: (64, 3, 28, 28) -> flattened = 3 x 28 x 28 = 2352 inputs. TinyNet's first layer must be Linear(2352, 128), not 784.
- labels: (64, 1), but CrossEntropyLoss wants (batch,). Fix with .squeeze() at loss time.
- labels dtype: int64 (correct, no fix needed).
- pixels: -1.0 to 1.0 (ToTensor -> 0..1, then Normalize -> ~-1..1).

In [3]:
import torch.nn as nn

class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2352, 128)   # 3 x 28 x 28 = 2352 (RGB), was 784 for grayscale
        self.fc2 = nn.Linear(128, 7)      # 7 classes, unchanged

    def forward(self, x):
        x = x.reshape(x.size(0), -1)      # (64, 3, 28, 28) -> (64, 2352)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = TinyNet()
print(model)

TinyNet(
  (fc1): Linear(in_features=2352, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=7, bias=True)
)


## TinyNet updated for RGB
- First layer changed: Linear(784, 128) -> Linear(2352, 128), because RGB input is 3 x 28 x 28 = 2352.
- 128 = chosen hidden width (a design choice). Input 2352 and output 7 are forced by the data.
- Flatten idiom `x.reshape(x.size(0), -1)` handles the shape change automatically.

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:           # loop over batches, not all data at once
        labels = labels.squeeze()                 # (64, 1) -> (64,) for CrossEntropyLoss
        logits = model(images)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg = running_loss / len(train_loader)
    print(f"epoch {epoch:2d}  train loss {avg:.4f}")

epoch  0  train loss 1.0046
epoch  1  train loss 0.9168
epoch  2  train loss 0.8822
epoch  3  train loss 0.8462
epoch  4  train loss 0.8186
epoch  5  train loss 0.7930
epoch  6  train loss 0.7747
epoch  7  train loss 0.7541
epoch  8  train loss 0.7397
epoch  9  train loss 0.7193


## Training on real data
Same 5-line loop as Phase 0, now looping over batches (110 per epoch, ~1 update each).
- `labels.squeeze()`: (64, 1) -> (64,) for CrossEntropyLoss.
- `model.train()`: sets training mode (habit; pairs with model.eval() later).
- Falling train loss is NOT proof of success here - we check validation next (overfitting lesson from Phase 0).

In [5]:
model.eval()   # evaluation mode (pairs with model.train())

correct = 0
total = 0
per_class_correct = [0] * 7
per_class_total = [0] * 7

with torch.no_grad():                     # no gradients needed when just evaluating
    for images, labels in val_loader:
        labels = labels.squeeze()
        logits = model(images)
        preds = logits.argmax(dim=1)      # pick the class with the highest score

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        for c in range(7):
            mask = (labels == c)
            per_class_total[c] += mask.sum().item()
            per_class_correct[c] += (preds[mask] == c).sum().item()

print(f"overall val accuracy: {100*correct/total:.1f}%\n")
for c in range(7):
    recall = 100 * per_class_correct[c] / per_class_total[c] if per_class_total[c] else 0
    print(f"class {c}: {per_class_correct[c]:3d}/{per_class_total[c]:3d} caught  ({recall:4.1f}% recall)")

overall val accuracy: 72.0%

class 0:   5/ 33 caught  (15.2% recall)
class 1:  29/ 52 caught  (55.8% recall)
class 2:  29/110 caught  (26.4% recall)
class 3:   0/ 12 caught  ( 0.0% recall)
class 4:  21/111 caught  (18.9% recall)
class 5: 635/671 caught  (94.6% recall)
class 6:   3/ 14 caught  (21.4% recall)


## Baseline results: accuracy hides the real story
Overall val accuracy: 72% - but this is misleading.
- Class 5 (67% of data): 94.6% recall. Rare classes: class 3 = 0%, class 4 (melanoma) = 18.9%.
- The model hits 72% mostly by nailing the majority class, NOT by learning lesions well.
- It has learned "when unsure, guess nevi" - the imbalance trap.
- Dangerous because failures concentrate on rare classes, including melanoma (misses 81%).
- Confirms: accuracy is misleading on imbalanced data; per-class recall is the metric that matters.
- Next: fight imbalance (weighted loss) and measure if rare-class recall improves.